## Title

### By:
Author Name

### Date:
2024-MM-DD

### Description:

General description of the notebook


## 📚 Import  libraries

In [4]:
# base libraries for data science
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

logging.basicConfig(level=logging.WARNING)

## 💾 Load data

In [ ]:
# data directory path
DATA_DIR = Path.cwd().resolve().parents[1] / "data"

churn_df = pd.read_csv(DATA_DIR / "01_raw/churn_raw_selected.csv")

print("=" * 80)
print("CARGA DE DATOS - INSPECCIÓN INICIAL")
print("=" * 80)
print(f"\nDimensiones: {churn_df.shape[0]} filas x {churn_df.shape[1]} columnas")
print(f"\nNombres de columnas:\n{list(churn_df.columns)}")
print(f"\nTipos de datos actuales:\n{churn_df.dtypes}")
print(f"\nValores nulos por columna:\n{churn_df.isnull().sum()}")
print(f"\nPorcentaje de valores nulos:\n{(churn_df.isnull().sum() / len(churn_df) * 100).round(2)}")
print(f"\nPrimeras filas:\n{churn_df.head()}")

## 🧹 Data Cleaning and Type Conversion

### Step 1: Detect Null Values and Fix Data Issues

In [ ]:
# Detect null values and fix data issues
print("=" * 80)
print("DETECCIÓN DE VALORES NULOS Y PROBLEMAS EN LOS DATOS")
print("=" * 80)

# TotalCharges has blank spaces ' ' that should be NaN
churn_df["TotalCharges"] = churn_df["TotalCharges"].replace(r"^\s*$", np.nan, regex=True)
churn_df["TotalCharges"] = pd.to_numeric(churn_df["TotalCharges"], errors="coerce")

# Check for null values by column
null_counts = churn_df.isnull().sum()
null_percentage = (null_counts / len(churn_df) * 100).round(2)

print("\nValores nulos encontrados:")
if null_counts.sum() == 0:
    print("✓ No hay valores nulos en el dataset")
else:
    null_info = pd.DataFrame(
        {
            "Column": null_counts.index,
            "Null_Count": null_counts.values,
            "Percentage": null_percentage.values,
        }
    )
    null_info = null_info[null_info["Null_Count"] > 0].sort_values("Null_Count", ascending=False)
    print(null_info)

print(f"\nTotal de valores nulos: {null_counts.sum()}")
print("\nNota: La imputación de valores nulos se realizará en la etapa de Feature Engineering.")

In [ ]:
print("=" * 80)
print("CONVERSIÓN DE TIPOS DE DATOS")
print("=" * 80)

# Display current types
print("\nTipos de datos actuales:")
print(churn_df.dtypes)

# Define type conversions
type_conversions = {
    # Numeric columns
    "tenure": "int32",
    "MonthlyCharges": "float32",
    "TotalCharges": "float64",
    # Boolean-like columns
    "SeniorCitizen": "int8",
    "Partner": "category",
    "Dependents": "category",
    "PaperlessBilling": "category",
    "Churn": "category",
    # Service columns
    "InternetService": "category",
    "OnlineSecurity": "category",
    "OnlineBackup": "category",
    "DeviceProtection": "category",
    "TechSupport": "category",
    "StreamingTV": "category",
    "StreamingMovies": "category",
    "MultipleLines": "category",
    # Contract and payment
    "Contract": "category",
    "PaymentMethod": "category",
}

# Convert types
for col, dtype in type_conversions.items():
    if col in churn_df.columns:
        try:
            if dtype == "category":
                churn_df[col] = churn_df[col].astype(dtype)
                unique_vals = churn_df[col].nunique()
                print(f"  ✓ {col}: convertido a 'category' ({unique_vals} categorías únicas)")
            else:
                churn_df[col] = churn_df[col].astype(dtype)
                print(f"  ✓ {col}: convertido a '{dtype}'")
        except Exception as e:
            logging.warning(f"Error converting {col} to {dtype}: {e}")
            print(f"  ✗ {col}: Error - {e}")

print("\nTipos de datos finales:")
print(churn_df.dtypes)
print(
    f"\nTamaño en memoria después de conversión: {churn_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

## 💾 Schema Extraction and Parquet Export

In [ ]:
print("=" * 80)
print("EXTRACCIÓN DEL SCHEMA Y EXPORTACIÓN A PARQUET")
print("=" * 80)

# Convert Pandas DataFrame to PyArrow Table to get schema
table = pa.Table.from_pandas(churn_df)
schema = table.schema

print("\nSchema del dataset:")
print(schema)

# Save schema as text file for reference
output_dir = DATA_DIR / "02_intermediate"
output_dir.mkdir(exist_ok=True, parents=True)

schema_file = output_dir / "churn_cleaned_schema.txt"
with open(schema_file, "w") as f:
    f.write("SCHEMA - CHURN CLEANED DATASET\n")
    f.write("=" * 80 + "\n\n")
    f.write(str(schema))
    f.write("\n\n" + "=" * 80 + "\n")
    f.write("COLUMN DETAILS:\n")
    f.write("=" * 80 + "\n")
    for i, field in enumerate(schema, 1):
        f.write(f"{i}. {field.name}: {field.type}\n")

print(f"\n✓ Schema guardado en: {schema_file}")

# Save cleaned data to Parquet
parquet_file = output_dir / "churn_cleaned.parquet"
pq.write_table(table, parquet_file, compression="snappy")

# Verify parquet file
parquet_info = pq.read_table(parquet_file)
print(f"\n✓ Datos limpios guardados en Parquet: {parquet_file}")
print(f"  - Filas: {parquet_info.num_rows}")
print(f"  - Columnas: {parquet_info.num_columns}")
print(f"  - Tamaño del archivo: {parquet_file.stat().st_size / 1024**2:.2f} MB")
print("  - Compresión: snappy")

# Summary statistics
print("\n" + "=" * 80)
print("RESUMEN DE LA LIMPIEZA Y CONVERSIÓN")
print("=" * 80)
print("Dataset Original: 7,043 filas x 18 columnas")
print(f"Dataset Final: {churn_df.shape[0]} filas x {churn_df.shape[1]} columnas")
print("Valores nulos eliminados: 0")
print(f"Tipos de datos convertidos: {len(type_conversions)}")
print("\nArchivos generados:")
print(f"  1. {parquet_file.name}")
print(f"  2. {schema_file.name}")

## 📊 Analysis of Results and Conclusions

### Data Cleaning Summary
- **Null Values**: Detected and reported (TotalCharges blank spaces converted to NaN). Imputation will be done in Feature Engineering.
- **Data Types**: Converted to appropriate types (int8, int32, float32, float64, category)
- **Memory Optimization**: Reduced memory footprint through efficient type assignments
- **Schema**: Extracted and validated using PyArrow

### Output Format
- **Format**: Apache Parquet with Snappy compression
- **Location**: `data/02_intermediate/churn_cleaned.parquet`
- **Schema File**: `data/02_intermediate/churn_cleaned_schema.txt`
- **Advantages**: 
  - Efficient columnar storage format
  - Preserves data types and schema information
  - Better performance for analytical queries
  - Language-independent format

### Data Quality Improvements
- Numeric columns optimized for storage (float32/float64)
- Categorical columns reduce memory and enable efficient encoding
- Consistent data types across all columns
- Schema preservation enables reproducibility

## 💡 Proposals and Ideas for Next Steps

### 1. **Exploratory Data Analysis (EDA)**
   - Univariate analysis: Distributions of each variable
   - Bivariate analysis: Relationships between features and churn
   - Multivariate analysis: Interaction patterns between features
   - Identify outliers and unusual patterns

### 2. **Feature Engineering**
   - Create interaction terms between key features
   - Calculate customer lifetime value (CLV)
   - Tenure-based segments (new, loyal, at-risk)
   - Service adoption rate
   - Charge ratio features (Monthly/Total)

### 3. **Class Imbalance Handling**
   - Evaluate current churn rate distribution
   - Apply SMOTE, ADASYN, or other sampling techniques
   - Consider class weights in model training
   - Use stratified sampling for train/test split

### 4. **Feature Selection**
   - Correlation analysis with target variable
   - Chi-square tests for categorical features
   - Feature importance using tree-based models
   - Dimensionality reduction if needed

### 5. **Model Development**
   - Baseline models: Logistic Regression, Decision Tree
   - Advanced models: Random Forest, XGBoost, LightGBM
   - Ensemble methods combining multiple models
   - Hyperparameter tuning using cross-validation

### 6. **Model Evaluation**
   - Use appropriate metrics: Precision, Recall, F1, AUC-ROC
   - Stratified K-fold cross-validation
   - Confusion matrix analysis
   - Feature importance interpretation

## 📖 References